# Memory 

# import

In [1]:
from dotenv import load_dotenv
print(load_dotenv())

from langchain_openai import ChatOpenAI

True


In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
from typing import Annotated
from typing_extensions import TypedDict

In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# 🟦 메모리를 사용하지 않는 경우

# 상태 정의

In [5]:
# 현재 대화 상태를 담는 객체 

In [6]:
class State(TypedDict):
    messages: Annotated[list[str], add_messages]   # "이건 list[str] 타입이야, 이를 업데이트 할때는 add_messages 를 사용해라"
            # add_messages 를 사용하면 기존 message 들를 '덮어쓰기' 하는게 아니라 '추가 append' 동작.

    

# 그래프 생성

In [8]:
workflow = StateGraph(State)

def generate(state: State):
    return {"messages": [model.invoke(state['messages'])]}

workflow.add_node("generate", generate)
workflow.add_edge(START, "generate")
workflow.add_edge("generate", END)

graph = workflow.compile()

In [9]:
from langchain_core.messages.human import HumanMessage

# 챗봇동작

In [10]:
while True:
    user_input = input("You\t:")

    if user_input in ["exit", "quit", "q"]: break

    # stream_mode= 를 'messages' 가 아니라 'values' 로 설정
    for event in graph.stream({"messages": [HumanMessage(user_input)]}, stream_mode="values"):
        event['messages'][-1].pretty_print()

    print(f"\n현재 메세지 개수: {len(event['messages'])}\n----------------------\n")        

You	: 난 뽀로로야


================================ Human Message =================================

난 뽀로로야
================================== Ai Message ==================================

안녕하세요, 뽀로로! 어떻게 도와드릴까요?

현재 메세지 개수: 2
----------------------



You	: 내 이름이 뭐지?


================================ Human Message =================================

내 이름이 뭐지?
================================== Ai Message ==================================

죄송하지만 제공하신 정보만으로는 당신의 이름을 알 수 없습니다. 이름을 알려주시면 도움이 필요한 부분에 대해 더 잘 도와드릴 수 있을 것입니다.

현재 메세지 개수: 2
----------------------



You	: q


# 🟦 메모리 사용

In [11]:
# ✅ langgraph 에서 제공하는 MemorySaver 

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [12]:
# ✅ thread_id 설정 
#   일종의 대화방 id (?) <- 임의의 문자열로 설정

config = {"configurable": {'thread_id': "abcd"}}


In [13]:
# ✅ compile(checkpointer=) 에 설정
graph = workflow.compile(checkpointer=memory)

# 챗봇동작

In [15]:
while True:
    user_input = input("You\t:")

    if user_input in ["exit", "quit", "q"]: break

    # stream_mode= 를 'messages' 가 아니라 'values' 로 설정
    for event in graph.stream(
            {"messages": [HumanMessage(user_input)]}, 
            config, #✅ config 전달!
            stream_mode="values"):
        event['messages'][-1].pretty_print()

    print(f"\n현재 메세지 개수: {len(event['messages'])}\n----------------------\n")        

You	: 난 뽀로로야


================================ Human Message =================================

난 뽀로로야
================================== Ai Message ==================================

안녕! 뽀로로라니 반가워. 오늘은 어떤 모험을 떠날 거니? 도움이 필요하면 언제든지 말해 줘.

현재 메세지 개수: 2
----------------------



You	: 내 이름은 뭐지?


================================ Human Message =================================

내 이름은 뭐지?
================================== Ai Message ==================================

당신이 '뽀로로'라고 소개했으니, 지금 대화에서는 당신을 뽀로로라고 부르면 될 것 같아요. 다른 이름이 있거나 원하는 이름이 있다면 알려주세요!

현재 메세지 개수: 4
----------------------



You	: q
